In [1]:
import pandas as pd

# Load the blood cell anomaly detection dataset
df = pd.read_csv('data/dataset/blood_cell_anomaly_detection.csv')

# Display basic information about the dataset
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
print("\nFirst 5 rows:")
df.head()

Dataset shape: (5880, 36)
Columns: ['cell_id', 'cell_type', 'anomaly_label', 'disease_category', 'cell_diameter_um', 'nucleus_area_pct', 'chromatin_density', 'cytoplasm_ratio', 'circularity', 'eccentricity', 'granularity_score', 'lobularity_score', 'membrane_smoothness', 'cell_area_px', 'perimeter_px', 'mean_r', 'mean_g', 'mean_b', 'stain_intensity', 'patient_age_group', 'patient_sex', 'wbc_count_per_ul', 'rbc_count_millions_per_ul', 'hemoglobin_g_dl', 'hematocrit_pct', 'platelet_count_per_ul', 'mcv_fl', 'mchc_g_dl', 'dataset_source', 'staining_protocol', 'microscope_model', 'magnification_x', 'image_resolution_px', 'cytodiffusion_anomaly_score', 'cytodiffusion_classification_confidence', 'labeller_confidence_score']

First 5 rows:


,cell_id,cell_type,anomaly_label,disease_category,cell_diameter_um,nucleus_area_pct,chromatin_density,cytoplasm_ratio,circularity,eccentricity,...,mcv_fl,mchc_g_dl,dataset_source,staining_protocol,microscope_model,magnification_x,image_resolution_px,cytodiffusion_anomaly_score,cytodiffusion_classification_confidence,labeller_confidence_score
0,CELL_005371,Hypersegmented_Neutrophil,1,Infection,15.18,58.8,0.542,0.301,0.563,0.529,...,85.5,31.4,CytoData,Giemsa,Zeiss_Axio,100,224,0.7649,0.5726,0.5670
1,CELL_005300,Hypersegmented_Neutrophil,1,Infection,16.47,73.6,0.583,0.365,0.859,0.443,...,92.5,35.0,PBC_Dataset,Wright,Zeiss_Axio,100,224,0.8472,0.7150,0.7273
2,CELL_000200,Neutrophil,0,Normal_WBC,13.41,55.5,0.448,0.376,0.781,0.407,...,76.3,33.0,CytoData,Wright,Leica_DM2000,100,512,0.0313,0.9225,0.9623
3,CELL_003269,Normal_RBC,0,Normal_RBC,7.36,0.0,0.000,1.000,0.880,0.167,...,92.3,32.5,CytoData,Wright,Leica_DM2000,100,512,0.1293,0.9180,0.8652
4,CELL_003505,Normal_RBC,0,Normal_RBC,7.53,0.0,0.000,1.000,1.000,0.158,...,83.9,33.4,CytoData,Wright,Olympus_BX51,100,224,0.1418,0.9697,0.8898


In [2]:
# Drop unnecessary columns
columns_to_drop = ['cell_area_px', 'perimeter_px', 'cell_id', 'magnification_x', 'image_resolution_px', 'labeller_confidence_score', 'cytodiffusion_classification_confidence', 'microscope_model', 'stain_intensity', 'mean_r', 'mean_g', 'mean_b', 'dataset_source', 'staining_protocol', 'cytodiffusion_anomaly_score']
df.drop(columns=columns_to_drop, inplace=True)

print("Columns after dropping:", list(df.columns))
print("New shape:", df.shape)

Columns after dropping: ['cell_type', 'anomaly_label', 'disease_category', 'cell_diameter_um', 'nucleus_area_pct', 'chromatin_density', 'cytoplasm_ratio', 'circularity', 'eccentricity', 'granularity_score', 'lobularity_score', 'membrane_smoothness', 'patient_age_group', 'patient_sex', 'wbc_count_per_ul', 'rbc_count_millions_per_ul', 'hemoglobin_g_dl', 'hematocrit_pct', 'platelet_count_per_ul', 'mcv_fl', 'mchc_g_dl']
New shape: (5880, 21)


In [3]:
# Prepare feature matrix X and labels y1 (anomaly) and y2 (cell type)
from pathlib import Path
import numpy as np
import pandas as pd

# y1: anomaly label, y2: cell type
y1 = df['anomaly_label']
y2 = df['cell_type']

# X: all columns except the two label columns
X = df.drop(columns=['anomaly_label', 'cell_type'])

# encode y2 as integer codes for modeling (optional)
y2_encoded = y2.astype('category').cat.codes

# Save processed splits
out_dir = Path('data/processed')
out_dir.mkdir(parents=True, exist_ok=True)
X.to_csv(out_dir / 'X.csv', index=False)
y1.to_csv(out_dir / 'y1.csv', index=False)
pd.Series(y2_encoded, name='cell_type_code').to_csv(out_dir / 'y2_encoded.csv', index=False)
# also save compact numpy archive
np.savez_compressed(out_dir / 'dataset_split.npz', X=X.values, y1=y1.values, y2=y2_encoded.values)
print('Saved X, y1, y2_encoded to', out_dir)

Saved X, y1, y2_encoded to data\processed


In [ ]:
from sklearn.model_selection import train_test_split

# Create stratified train/validation/test split to maintain class ratios for both y1 and y2
# Combine y1 and y2 into a stratification key to preserve both class distributions
strat_key = y1.astype(str) + "_" + y2_encoded.astype(str)

# First split: 70% train, 30% temp (validation + test)
X_train, X_temp, y1_train, y1_temp, y2_train, y2_temp, strat_train, strat_temp = train_test_split(
    X, y1, y2_encoded, strat_key, test_size=0.3, random_state=42, stratify=strat_key
)

# Second split: split temp into 50% validation, 50% test (which is 15% each of total)
X_val, X_test, y1_val, y1_test, y2_val, y2_test = train_test_split(
    X_temp, y1_temp, y2_temp, test_size=0.5, random_state=42, stratify=strat_temp
)

print(f"Train split: X_train shape = {X_train.shape}, y1_train shape = {y1_train.shape}")
print(f"Val split:   X_val shape = {X_val.shape}, y1_val shape = {y1_val.shape}")
print(f"Test split:  X_test shape = {X_test.shape}, y1_test shape = {y1_test.shape}")

# Verify class balance across splits
print("\n--- Class Distribution ---")
for split_name, y1_s, y2_s in [
    ('Train', y1_train, y2_train),
    ('Val', y1_val, y2_val),
    ('Test', y1_test, y2_test)
]:
    print(f"\n{split_name}:")
    print(f"  y1 (anomaly) value counts:\n{y1_s.value_counts().sort_index()}")
    print(f"  y2 (cell type) unique: {len(y2_s.unique())} types")

# Save splits
splits_dir = Path('data/splits')
splits_dir.mkdir(parents=True, exist_ok=True)

# Save as CSV
for split, (X_s, y1_s, y2_s) in [
    ('train', (X_train, y1_train, y2_train)),
    ('val', (X_val, y1_val, y2_val)),
    ('test', (X_test, y1_test, y2_test))
]:
    X_s.to_csv(splits_dir / f'X_{split}.csv', index=False)
    y1_s.to_csv(splits_dir / f'y1_{split}.csv', index=False)
    pd.Series(y2_s, name='cell_type_code').to_csv(splits_dir / f'y2_{split}.csv', index=False)

# Save as compressed numpy archive
np.savez_compressed(
    splits_dir / 'train_val_test_split.npz',
    X_train=X_train.values, y1_train=y1_train.values, y2_train=y2_train.values,
    X_val=X_val.values, y1_val=y1_val.values, y2_val=y2_val.values,
    X_test=X_test.values, y1_test=y1_test.values, y2_test=y2_test.values
)

print(f"\nSaved stratified train/val/test splits to {splits_dir}")